In [ ]:
import pandas as pd
import os
from dataframe_modify.add_positions import add_positions
from dataframe_modify.swap_perspective import swap_y
from dataframe_modify.swap_perspective import generalize_perspective_sb360
from dataframe_modify.grid_areas import add_grid_areas
from sequences.get_sequences import get_sequences, get_corners_sequences, get_sequences_by_possession
from sequences.find_fast_actions import find_fast_actions_forward_perspective
from fast_actions.get_dataframe import get_fast_action_row
from corners.metric import evaluate_corner
from corners.get_dataframe import get_df_row, prepare_matches_from_pkl

In [ ]:
input_name = 'esa_22_23_matches.pkl'
output_name = 'corners_xgb_22_23_more_columns_on_goal.pkl'

In [ ]:
params = {
    'pass_to_zone_I': -0.8,
    'pass_to_zone_II': 0.1,
    'pass_to_zone_III': 0.2,
    'pass_incomplete': -0.5,
    'pass_shot_assist': 0.4,
    'pass_goal_assist': 0.4,
    'pass_cross': 0.2,
    'pass_by_opponent': -0.5,
    'shot_goal': 2,
    'shot_saved': 1,
    'shot_blocked': 0.5,
    'shot_off_target': 0.5,
    'shot_post': 1,
    'shot_saved_to_post': 1,
    'shot_saved_off_target': 0.5,
    'clearance_by_opponent': -0.3,
    'ball_recovery_by_opponent': -0.3,
    'ball_recovery_by_own_team': 0.5,
    'foul_committed_by_own_team': -0.5,
    'foul_committed_by_own_team_with_card': -1,
    'foul_committed_by_opponent': 0.5,
    'foul_committed_by_opponent_with_penalty': 1,
}

In [ ]:
esa_matches = prepare_matches_from_pkl(f'data/{input_name}')

In [ ]:
sequences = get_sequences_by_possession(esa_matches)
# sequences = get_sequences(esa_matches)
corners = get_corners_sequences(sequences)

In [ ]:
corners =  [evaluate_corner(corner, params) for corner in corners]

In [ ]:
def df_print_corner(df):
    corner_df = df[['period', 'timestamp', 'type', 'team', 'player', 'event', 'corner_score']]
    return corner_df

In [ ]:
df_print_corner(corners[-1])#.to_csv('corner_2.csv', index=False)

In [ ]:
from corners.get_dataframe import get_df_row
xgb_df = pd.DataFrame()
xgb_df = xgb_df.append([get_df_row(corner, end_pass_zones=True, shot_on_goal=True) for corner in corners])

In [ ]:
len(xgb_df) #3192

In [ ]:
xgb_df

In [ ]:
xgb_df['shot_on_goal'].value_counts()


In [ ]:
xgb_df.to_pickle(f'data/{output_name}')